# Lesson 5: Sub-agents
So far one agent did everything. Two problems kept showing up:
- The news tool returned headlines that were **not about the company**.
- The agent stated facts **no tool gave it** (a dividend yield, an 'average P/E').

The fix is **sub-agents**: small agents that each do one focused job. The main agent hands work to them the same way it calls a tool.

The sub-agent code is in `stock_advisor/agents.py`. Run the cells top to bottom with **Shift+Enter**.

## Step 0: Setup

In [ ]:
import sys, inspect
sys.path.insert(0, '..')

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from IPython.display import display, Markdown

from stock_advisor.tools import get_recent_news, calculate_shares, buy_stock
from stock_advisor.agents import get_relevant_news, research_company, review, researcher, EVIDENCE, MODEL

load_dotenv()

## Step 1: Sub-agent #1, the news checker
Compare the old news tool with the new one. The new one fetches the same headlines, then a checking agent keeps only those about the company.

In [ ]:
print('OLD (no checking):')
print(get_recent_news.invoke({'ticker': 'NVDA'}))
print()
print('NEW (checked by a sub-agent):')
print(get_relevant_news.invoke({'ticker': 'NVDA'}))

## Step 2: How it is built
A sub-agent is a normal agent with a narrow job. This one has **no tools**. It reads a list of headlines and answers in a fixed shape (`NewsVerdict`). Then an ordinary tool wraps it, so the main agent can call it like any other tool.

Read the code:

In [ ]:
import stock_advisor.agents as A
print(inspect.getsource(A.NewsVerdict))
print(inspect.getsource(A.news_checker.__class__) if False else '')
print(inspect.getsource(get_relevant_news.func))

**Structured output.** `response_format=NewsVerdict` forces the agent to answer with a list of numbers and a reason, not free text. Your code can then read the answer safely. The result is in `result['structured_response']`.

**Imperfect.** The checker is a language model, so it can make mistakes. It sometimes keeps borderline headlines such as 'Top analyst reports for Apple, Philip Morris & Intel'.

## Step 3: Sub-agent #2, the researcher
The researcher owns all the research tools. We wrap the whole agent as one tool, `research_company`. The main agent will no longer see any research tool. It sees only this one.

In [ ]:
report = research_company.invoke({'request': 'Apple: current price, 6 month trend versus the market, and recent news'})
print(report)

## Step 4: The main agent as a manager
The main advisor has three tools: `research_company` (a whole sub-agent), `calculate_shares`, and `buy_stock` (still behind the human approval from Lesson 4). It never touches raw market data.

Why split it this way?
- The main agent's conversation stays **short**. It sees a 150-word report, not pages of raw numbers.
- Each agent has a **small job**, so it follows its rules better.
- You can test and improve each one separately.

In [ ]:
advisor = create_agent(
    model=MODEL,
    tools=[research_company, calculate_shares, buy_stock],
    system_prompt=(
        "You are a careful stock advisor. For any research, call research_company once per company. "
        "Use calculate_shares for all money math. Base your answer only on what your tools returned. "
        "If the user asks to buy, call buy_stock. Keep answers under 250 words. "
        "When you mention news, quote the headline exactly. Never say what it implies. "
        "Say this is research, not financial advice."
    ),
    middleware=[HumanInTheLoopMiddleware(interrupt_on={'buy_stock': True})],
    checkpointer=InMemorySaver(),
)

def ask(thread_id, text):
    EVIDENCE.clear()
    config = {'configurable': {'thread_id': thread_id}}
    result = advisor.invoke({'messages': [{'role': 'user', 'content': text}]}, config=config)
    return result, config

result, config = ask('demo', 'I have $5000 and I prefer low risk. Research Apple for me and tell me how many shares I could afford.')

print('Tools the main agent called:')
for m in result['messages']:
    if m.type == 'ai' and m.tool_calls:
        for c in m.tool_calls:
            print('  ', c['name'], c['args'])
display(Markdown(result['messages'][-1].content))

## Step 5: Sub-agent #3, the reviewer
The reviewer reads the **evidence** (raw tool results) and an **answer**, and lists every claim the evidence does not support.

First a test with an answer where we **planted** two made-up claims:

In [ ]:
evidence = ['AAPL: price 354.07, trailingPE 38.87', 'AAPL over 6mo: change +15.0%', 'S&P 500 over 6mo: change -4.2%']

good = 'Apple trades at $354.07 with a trailing P/E of 38.87. It rose 15.0% in 6 months while the S&P 500 fell 4.2%.'
bad = good + ' Apple pays a dividend of about 0.6%, and its P/E is above the market average of 22.'

for name, answer in [('GOOD ANSWER', good), ('ANSWER WITH MADE-UP CLAIMS', bad)]:
    r = review(answer, evidence)
    print(name, '->', r.verdict.upper(), r.unsupported_claims)

## Step 6: Review the real answer, and fix it automatically
The loop:
1. The advisor answers.
2. The reviewer checks the answer against the evidence.
3. If it fails, the advisor gets the list of unsupported claims and rewrites its answer once.

The evidence is everything the tools returned, plus what **you** said (so your own budget is not flagged).

In [ ]:
def evidence_from(result):
    tool_results = [m.content for m in result['messages'] if m.type == 'tool' and m.name != 'research_company']
    user_said = 'USER SAID: ' + result['messages'][0].content
    return EVIDENCE + tool_results + [user_said]


def check(result):
    report = review(result['messages'][-1].content, evidence_from(result))
    print('REVIEW:', report.verdict.upper())
    for claim in report.unsupported_claims:
        print('  unsupported:', claim)
    return report


def ask_with_review(thread_id, text, max_revisions=1):
    result, config = ask(thread_id, text)
    report = check(result)
    for _ in range(max_revisions):
        if report.verdict == 'pass':
            break
        feedback = ('A fact checker found claims with no support in your tool results:\n- '
                    + '\n- '.join(report.unsupported_claims)
                    + '\nRewrite your answer without them. If you lack data, say so.')
        print('Asking the advisor to revise...')
        result = advisor.invoke({'messages': [{'role': 'user', 'content': feedback}]}, config=config)
        report = check(result)
    display(Markdown(result['messages'][-1].content))
    return result

In [ ]:
# First, check the answer from Step 4 as it was:
_ = check(result)

In [ ]:
_ = ask_with_review('review-demo', 'I have $3000 and I like safe investments. Is Microsoft a good fit? How many shares could I buy?')

## Things to notice
1. **A sub-agent is just an agent wrapped as a tool.** The main agent does not know or care that `research_company` is a whole agent inside.
2. **Structured output** (`response_format=...`) is how you get reliable answers from an agent that your code must read.
3. **Each sub-agent costs time and money.** The news checker, the researcher and the reviewer are extra model calls. Check how long Step 6 took compared with Lesson 3.
4. **The reviewer is a model too.** It can miss a bad claim or flag a good one. It lowers the risk. It does not remove it.
5. **Evidence matters.** The reviewer can only catch claims if it can see what the tools actually returned. That is why `research_company` saves raw results in `EVIDENCE`.
6. **The human step still comes last.** None of this replaces the approval from Lesson 4. Buying is still gated.

## Exercise
1. Run Step 1 for `AAPL` and `TSLA`. Which headlines did the checker keep? Do you agree with each one?
2. In Step 5, add a third planted claim to `bad` (for example, `'Apple has no debt.'`). Does the reviewer catch it?
3. Run `ask_with_review` with a question about **two** companies. How many times did the main agent call `research_company`?
4. Think about this: which would you rather have, a reviewer that sometimes flags good answers, or one that sometimes misses bad ones? Why?

In [ ]:
# your code here